# 04 — Training Model Skrining Insomnia (JST / MLPClassifier)

Melatih **Model 1 (Skrining Gejala Insomnia)** menggunakan 4 fitur:
`[Sleep Duration, Quality of Sleep, Physical Activity Level, Stress Level]`
dan target biner (`0=Normal/Lainnya`, `1=Insomnia`).

Arsitektur: `MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu', solver='adam', alpha=0.01, max_iter=3000, random_state=42)` + `StandardScaler`.

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

BASE_DIR = Path("..") if Path("../dataset").exists() else Path(".")
DATA_PATH = BASE_DIR / "dataset" / "processed" / "sleep_cleaned.csv"
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = ["Sleep Duration", "Quality of Sleep", "Physical Activity Level", "Stress Level"]
TARGET = "target"

In [ ]:
df = pd.read_csv(DATA_PATH)
X = df[FEATURES].values
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

In [ ]:
model = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation="relu",
    solver="adam",
    alpha=0.01,
    learning_rate_init=0.001,
    max_iter=3000,
    random_state=42,
)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f} (n_iter={model.n_iter_}, loss={model.loss_:.4f})")
print(classification_report(y_test, y_pred, target_names=["Normal", "Insomnia"]))

In [ ]:
joblib.dump(model, MODEL_DIR / "sleep_model.keras")
joblib.dump(scaler, MODEL_DIR / "sleep_scaler.pkl")
print("Model & scaler insomnia tersimpan di:", MODEL_DIR.resolve())